In [1]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

c:\GitRepo\handson-mlp\.venv\Lib\site-packages\torch\cuda\__init__.py:187: UserWarning: cudaGetDeviceCount() returned cudaErrorNotSupported, likely using older driver or on CPU machine (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:88.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")

print(imdb)
print(imdb["train"][0])

train_data = imdb["train"]
test_data = imdb["test"]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

In [3]:
from typing import TypedDict
from torch.utils.data import Dataset, DataLoader

class Imdb_dict(TypedDict):
    text: str
    label: int
    
class ImdbDataset(Dataset):
    def __init__(self, imdb_data:Imdb_dict):
        self.X = imdb_data["text"]
        self.y = imdb_data["label"]
        self.len:int = len(imdb_data)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self) -> int:
        return self.len
    
train_loader = DataLoader(ImdbDataset(train_data), batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(ImdbDataset(test_data), batch_size=32, pin_memory=True)

In [ ]:
from transformers import BertConfig, BertTokenizer, PreTrainedTokenizerBase

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
config = BertConfig(
    vocab_size=bert_tokenizer.vocab_size, hidden_size=128, num_hidden_layers=2,
    num_attention_heads=4, intermediate_size=512, max_position_embeddings=128
)

In [ ]:
class InhuBertEmbedding(nn.Module):
    def __init__(self, bert_config:BertConfig, dropout:float = 0.1):
        supre().__init__()
        self.word_embeddings = nn.Embedding(
            bert_config.vocab_size, bert_config.hidden_size)
        self.position_embeddins = nn.Embedding(
            bert_config.max_position_embeddings, bert_config.hidden_size)
        self.layer_norm = nn.LayerNorm(bert_config.hidden_size)
        self.dropout = nn.Dropout(dropout)
    
    
    def forward(self, input_ids:Tensor):
        seq_len = input_ids.size(1)
        position_ids = torch.arange(seq_len, device=input_ids.device).unsqueeze(0) #1d array -> 2d array
        
        embeddings = self.word_embeddings(input_ids) + self.position_embeddins(position_ids)
        embeddings = self.layer_norm(embeddings)
        embeddings = self.dropout(embeddings)
        
        return embeddings
        
class InhuMultiheadAttention(nn.Module):
    def __init__(self, head_num, embed_size, hidden_size):
        pass

class InhuBertSentimentAnal(nn.Module):
    def __init__(self, bert_config:BertConfig, tokenizer:PreTrainedTokenizerBase=bert_tokenizer):
        super().__init__()
        self.bert_config:BertConfig = bert_config
        self.tokenizer:PreTrainedTokenizerBase = tokenizer
        
        
        
        